# ANALISIS FRAUDE FINANCIERO Y CONTABLE

Este proyecto busca desarrollar un modelo de Machine Learning capaz de identificar observaciones financieras con alto riesgo de estar asociadas a fraude o manipulación contable, a partir de variables financieras históricas de empresas.



En auditoría, compliance o regulación financiera, revisar manualmente miles de empresas y períodos contables es costoso e ineficiente. Un modelo predictivo puede ayudar a priorizar casos de mayor riesgo, optimizando recursos y mejorando la detección temprana de irregularidades.

In [5]:
import pandas as pd

In [6]:
df=pd.read_csv('/Users/igna/git/ProjectoFinal_4Geeks/data/Cleaned_data_1995_2018.csv')
df.head()

,ID,CIQ_ID,Financial_Year,abee,ccss,cinv,crec,croa,issu,oleas,...,pstk,re,rect,sale,sstk,txp,txt,xint,prcc_f,AAER_ID
0,1,2207Q,FY1995,0.000000,0.015935,-0.002493,0.008126,0.000000,1,0,...,0.0,249.74100,121.79100,953.13700,0.000,0.000,45.87600,-77.50600,0.0,NaN
1,1,2207Q,FY1996,0.000000,0.062410,0.000518,-0.003805,-0.009743,1,0,...,0.0,227.63000,126.79400,980.25500,70.000,0.000,32.27200,-76.17900,0.0,NaN
2,1,2207Q,FY1997,0.000000,0.004411,-0.001632,0.001364,0.005372,1,0,...,0.0,221.62300,139.96000,998.36900,0.000,0.000,49.99900,-81.21500,0.0,NaN
3,2,2789Q,FY1995,0.000000,0.052965,-0.000658,-0.019898,-0.031917,1,1,...,0.0,441.55800,245.49000,2895.98200,0.000,5.855,22.85600,-38.31800,0.0,NaN
4,3,AACC,FY2004,-0.046628,0.658153,0.000000,-0.026577,-0.144680,0,1,...,0.0,37.45968,217.16817,214.75313,96.077,0.000,11.30129,-1.70907,0.0,NaN


In [7]:
df.shape

(87974, 120)

In [8]:
df.columns

Index(['ID', 'CIQ_ID', 'Financial_Year', 'abee', 'ccss', 'cinv', 'crec',
       'croa', 'issu', 'oleas',
       ...
       'pstk', 're', 'rect', 'sale', 'sstk', 'txp', 'txt', 'xint', 'prcc_f',
       'AAER_ID'],
      dtype='str', length=120)

In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 87974 entries, 0 to 87973
Columns: 120 entries, ID to AAER_ID
dtypes: float64(99), int64(19), str(2)
memory usage: 80.5 MB


En una primera inspección, el dataset presenta 87.974 observaciones y 120 variables, mayormente numéricas. Esto lo convierte en una base adecuada para aplicar técnicas de Machine Learning. Además, la presencia de variables financieras e identificadores sugiere que el conjunto de datos puede ser utilizado para modelar riesgo de fraude financiero o manipulación contable.

In [10]:
df[['ID', 'CIQ_ID', 'Financial_Year', 'AAER_ID']].head()

,ID,CIQ_ID,Financial_Year,AAER_ID
0,1,2207Q,FY1995,NaN
1,1,2207Q,FY1996,NaN
2,1,2207Q,FY1997,NaN
3,2,2789Q,FY1995,NaN
4,3,AACC,FY2004,NaN


In [11]:
df['AAER_ID'].isna().sum()

np.int64(87399)

La columna AAER_ID presenta 87.399 valores nulos sobre un total de 87.974 observaciones, por lo que solo 575 registros contienen información en dicha variable. Esto sugiere que, en caso de utilizarse como base para definir la variable objetivo, el problema sería altamente desbalanceado, con una proporción muy pequeña de casos positivos frente a los negativos.



In [12]:
df['AAER_ID'].value_counts(dropna=False).head(10)

AAER_ID
NaN       87399
3182.0       11
2994.0       11
3784.0       10
2847.0        9
2953.0        9
2558.0        9
3073.0        8
3045.0        8
2628.0        7
Name: count, dtype: int64

In [13]:
df['target_fraud'] = df['AAER_ID'].notna().astype(int)
df['target_fraud'].value_counts()

/var/folders/5b/yfj9mmxn2919wjkjy_npmqvm0000gn/T/ipykernel_3662/791654551.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['target_fraud'] = df['AAER_ID'].notna().astype(int)


target_fraud
0    87399
1      575
Name: count, dtype: int64

Se creo una columna llamada target_fraud, que ve la columna AAER_ID y si tiene valor pone un 1, y si no tiene valor pone un 0.

In [14]:
df['target_fraud'].value_counts(normalize=True)

target_fraud
0    0.993464
1    0.006536
Name: proportion, dtype: float64

Se definió la variable objetivo target_fraud a partir de la presencia o ausencia de valores en AAER_ID, asignando valor 1 a los registros con AAER_ID no nulo y valor 0 a aquellos con AAER_ID nulo.
A partir de la construcción de la variable objetivo target_fraud, se observa un fuerte desbalance de clases: el 99.35% de las observaciones corresponde a la clase 0 y solo el 0.65% a la clase 1. Esto implica que el problema debe abordarse como una clasificación binaria desbalanceada, por lo que no será suficiente evaluar el desempeño del modelo únicamente con accuracy

In [15]:
df.describe()

,ID,abee,ccss,cinv,crec,croa,issu,oleas,rsst,soft,...,re,rect,sale,sstk,txp,txt,xint,prcc_f,AAER_ID,target_fraud
count,87974.000000,87974.000000,87974.000000,87974.000000,87974.000000,87974.000000,87974.000000,87974.000000,87974.000000,87974.000000,...,8.797400e+04,8.797400e+04,8.797400e+04,8.797400e+04,87974.000000,87974.000000,8.797400e+04,8.797400e+04,575.000000,87974.000000
mean,4686.705674,-0.088216,0.186542,-0.001647,0.007318,0.008937,0.836224,0.872849,0.043087,0.581637,...,1.831668e+03,1.429331e+03,7.719005e+03,1.719621e+02,78.193644,193.669410,-1.590716e+02,4.842894e+07,2827.756522,0.006536
std,2761.083114,0.410830,0.593149,0.026212,0.069019,0.177500,0.370074,0.333144,0.216706,0.275172,...,3.517888e+04,4.494907e+04,2.431917e+05,1.294505e+04,1999.273905,4067.914219,6.702432e+03,8.939626e+09,740.489555,0.080581
min,1.000000,-2.879165,-0.981900,-0.121349,-0.314628,-0.695967,0.000000,0.000000,-0.770905,0.033898,...,-8.179800e+05,-5.600000e-02,-8.106480e+05,0.000000e+00,-29507.941980,-34831.000000,-1.033361e+06,0.000000e+00,1017.000000,0.000000
25%,2268.000000,-0.110360,0.000000,-0.002344,-0.003799,-0.017312,1.000000,1.000000,-0.020384,0.360974,...,-3.529850e+01,9.809210e+00,1.122973e+02,2.421325e-01,0.000000,0.018000,-3.600000e+01,0.000000e+00,2351.000000,0.000000
50%,4731.000000,-0.008386,0.069692,0.000000,0.002109,0.000000,1.000000,1.000000,0.023735,0.613001,...,4.916850e+01,5.439500e+01,4.434340e+02,3.250000e+00,0.000000,7.524500,-4.501500e+00,6.125000e+00,2885.000000,0.000000
75%,7081.000000,0.027691,0.216099,0.001006,0.022402,0.018155,1.000000,1.000000,0.087910,0.822331,...,3.655998e+02,2.322995e+02,1.716140e+03,2.543036e+01,3.577750,40.200000,-6.848000e-02,2.255000e+01,3271.500000,0.000000
max,9345.000000,0.895199,4.406284,0.100886,0.286244,1.172544,1.000000,1.000000,1.045389,0.981694,...,3.931265e+06,7.248251e+06,3.840189e+07,2.786302e+06,250454.000000,413061.218600,5.460000e+02,2.070000e+12,4275.000000,1.000000


In [16]:
df.drop("ID", axis = 1).duplicated().sum()

np.int64(0)

# EDA

In [17]:
columnas_modelo = [
    'Financial_Year',
    'sale',
    'ni',
    'at',
    'lt',
    'che',
    'rect',
    'invt',
    'cogs',
    'txt',
    'xint',
    'prcc_f',
    'target_fraud'
]

df_modelo = df[columnas_modelo]
df_modelo.head()

,Financial_Year,sale,ni,at,lt,che,rect,invt,cogs,txt,xint,prcc_f,target_fraud
0,FY1995,953.13700,81.76800,2620.89600,1652.96900,5.69100,121.79100,50.893,328.9560,45.87600,-77.50600,0.0,0
1,FY1996,980.25500,58.76700,2670.76200,1809.41800,15.27800,126.79400,53.497,372.3240,32.27200,-76.17900,0.0,0
2,FY1997,998.36900,74.40500,2723.88400,1874.85100,17.22400,139.96000,50.135,326.3150,49.99900,-81.21500,0.0,0
3,FY1995,2895.98200,30.29700,1878.78300,1409.61300,135.65000,245.49000,40.358,2589.9240,22.85600,-38.31800,0.0,0
4,FY2004,214.75313,19.07852,252.50572,55.32555,14.20458,217.16817,0.000,56.9486,11.30129,-1.70907,0.0,0


In [18]:
df_modelo['Financial_Year'] = df_modelo['Financial_Year'].str.replace('FY', '').astype(int)

In [19]:
df_modelo.head()

,Financial_Year,sale,ni,at,lt,che,rect,invt,cogs,txt,xint,prcc_f,target_fraud
0,1995,953.13700,81.76800,2620.89600,1652.96900,5.69100,121.79100,50.893,328.9560,45.87600,-77.50600,0.0,0
1,1996,980.25500,58.76700,2670.76200,1809.41800,15.27800,126.79400,53.497,372.3240,32.27200,-76.17900,0.0,0
2,1997,998.36900,74.40500,2723.88400,1874.85100,17.22400,139.96000,50.135,326.3150,49.99900,-81.21500,0.0,0
3,1995,2895.98200,30.29700,1878.78300,1409.61300,135.65000,245.49000,40.358,2589.9240,22.85600,-38.31800,0.0,0
4,2004,214.75313,19.07852,252.50572,55.32555,14.20458,217.16817,0.000,56.9486,11.30129,-1.70907,0.0,0


In [28]:
df_modelo.info()

<class 'pandas.DataFrame'>
RangeIndex: 87974 entries, 0 to 87973
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Financial_Year  87974 non-null  int64  
 1   sale            87974 non-null  float64
 2   ni              87974 non-null  float64
 3   at              87974 non-null  float64
 4   lt              87974 non-null  float64
 5   che             87974 non-null  float64
 6   rect            87974 non-null  float64
 7   invt            87974 non-null  float64
 8   cogs            87974 non-null  float64
 9   txt             87974 non-null  float64
 10  xint            87974 non-null  float64
 11  prcc_f          87974 non-null  float64
 12  target_fraud    87974 non-null  int64  
dtypes: float64(11), int64(2)
memory usage: 8.7 MB


In [20]:
X = df_modelo.drop(columns='target_fraud')
y = df_modelo['target_fraud']

In [29]:
y.value_counts()

target_fraud
0    87399
1      575
Name: count, dtype: int64

## Paso extra de EDA: fraude por anio
Antes de modelar, conviene revisar si la tasa de fraude cambia segun el anio fiscal. Eso nos ayuda a detectar periodos con mayor concentracion de casos positivos y a interpretar mejor el contexto temporal del problema.

In [ ]:
fraude_por_anio = (
    df_modelo.groupby('Financial_Year')['target_fraud']
    .agg(total_observaciones='count', casos_fraude='sum', tasa_fraude='mean')
    .sort_index()
)
fraude_por_anio.tail(10)


## Paso extra de EDA: faltantes en variables del modelo
Tambien revisamos el porcentaje de valores faltantes en las variables elegidas para el baseline. Este paso justifica si alcanza con imputacion simple o si alguna variable requiere un tratamiento especial antes de entrenar.

In [ ]:
faltantes_modelo = (df_modelo.isna().mean() * 100).sort_values(ascending=False)
faltantes_modelo[faltantes_modelo > 0].to_frame('porcentaje_faltante').head(10)


El dataset no presenta valores nulos y está listo para modelado. Sin embargo, existe un desbalance extremo en la variable objetivo (99.34% clase 0 vs 0.66% clase 1), lo que implica que métricas como accuracy no serán adecuadas. Será necesario utilizar métricas como recall, precision y F1-score, además de aplicar técnicas de balanceo para mejorar la detección de fraudes.

In [21]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train.shape, X_test.shape, y_train.shape, y_test.shape

((70379, 12), (17595, 12), (70379,), (17595,))

## Modelado baseline

A partir del split inicial, entrenamos modelos baseline para un problema fuertemente desbalanceado. En este caso no alcanza con accuracy, por lo que vamos a comparar principalmente `average_precision`, `roc_auc`, `recall`, `precision` y `f1`.

In [22]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, f1_score, precision_recall_curve, precision_score, recall_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [23]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.1875,
    random_state=42,
    stratify=y_train
)

X_train.shape, X_val.shape, X_test.shape

((57182, 12), (13197, 12), (17595, 12))

In [24]:
def select_best_threshold(y_true, y_scores):
    precision, recall, thresholds = precision_recall_curve(y_true, y_scores)
    best_threshold = 0.5
    best_f1 = -1

    for idx, threshold in enumerate(thresholds):
        p = precision[idx]
        r = recall[idx]
        if p + r == 0:
            continue
        current_f1 = 2 * p * r / (p + r)
        if current_f1 > best_f1:
            best_f1 = current_f1
            best_threshold = threshold

    return float(best_threshold)

In [25]:
modelos = {
    'logistic_regression': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(class_weight='balanced', max_iter=2000, random_state=42))
    ]),
    'random_forest': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model', RandomForestClassifier(
            n_estimators=300,
            min_samples_leaf=5,
            class_weight='balanced_subsample',
            random_state=42,
            n_jobs=-1
        ))
    ])
}

In [26]:
resultados = []
modelos_entrenados = {}

for nombre, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    y_val_scores = modelo.predict_proba(X_val)[:, 1]
    threshold = select_best_threshold(y_val, y_val_scores)
    y_val_pred = (y_val_scores >= threshold).astype(int)

    resultados.append({
        'model': nombre,
        'threshold': threshold,
        'average_precision': average_precision_score(y_val, y_val_scores),
        'roc_auc': roc_auc_score(y_val, y_val_scores),
        'recall': recall_score(y_val, y_val_pred, zero_division=0),
        'precision': precision_score(y_val, y_val_pred, zero_division=0),
        'f1': f1_score(y_val, y_val_pred, zero_division=0)
    })

    modelos_entrenados[nombre] = modelo

resultados_df = pd.DataFrame(resultados).sort_values('average_precision', ascending=False)
resultados_df.round(4)

,model,threshold,average_precision,roc_auc,recall,precision,f1
1,random_forest,0.1331,0.0693,0.8019,0.2442,0.1296,0.1694
0,logistic_regression,0.7623,0.0129,0.6124,0.0233,0.0769,0.0357


In [27]:
mejor_fila = resultados_df.iloc[0]
mejor_modelo = modelos_entrenados[mejor_fila['model']]
mejor_threshold = mejor_fila['threshold']

y_test_scores = mejor_modelo.predict_proba(X_test)[:, 1]
y_test_pred = (y_test_scores >= mejor_threshold).astype(int)

print('Mejor modelo:', mejor_fila['model'])
print('Threshold elegido:', round(mejor_threshold, 4))
print('Test average_precision:', round(average_precision_score(y_test, y_test_scores), 4))
print('Test roc_auc:', round(roc_auc_score(y_test, y_test_scores), 4))
print('Test recall:', round(recall_score(y_test, y_test_pred, zero_division=0), 4))
print('Test precision:', round(precision_score(y_test, y_test_pred, zero_division=0), 4))
print('Test f1:', round(f1_score(y_test, y_test_pred, zero_division=0), 4))

Mejor modelo: random_forest
Threshold elegido: 0.1331
Test average_precision: 0.0946
Test roc_auc: 0.8458
Test recall: 0.2783
Test precision: 0.1212
Test f1: 0.1689


El baseline muestra que sí existe señal predictiva, especialmente en `RandomForest`, pero el problema sigue siendo difícil por el fuerte desbalance de clases. El siguiente paso natural es probar más variables financieras, construir ratios contables y ajustar hiperparámetros para mejorar la detección de la clase fraudulenta.